# YOLO 物件偵測訓練

In [2]:
PRETRAIN_MODEL = "yolo11n_costum.yaml"
LABEL_DIR = "/media/jianhua/HDD 1T/mot_annotations"
DATASET_DIR = "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT"
YOLO_LABEL_DIR = "/media/jianhua/HDD 1T/yolo_obj_detect"
DATA_YAML = "data.yaml"

## 轉換格式 COCO -> YOLO

In [ ]:
from ultralytics.data.converter import convert_coco

convert_coco(
    labels_dir=LABEL_DIR,
    save_dir=YOLO_LABEL_DIR
)

In [ ]:
import os
import shutil

src_labels = os.path.join(YOLO_LABEL_DIR, "labels")
dst_labels = os.path.join(DATASET_DIR, "labels")

# 1️⃣ 如果 DATASET_DIR 已有 labels 就刪掉
if os.path.exists(dst_labels):
    shutil.rmtree(dst_labels)

# 2️⃣ 搬移 labels 資料夾
shutil.move(src_labels, dst_labels)

# 3️⃣ 刪掉整個 YOLO_LABEL_DIR
if os.path.exists(YOLO_LABEL_DIR):
    shutil.rmtree(YOLO_LABEL_DIR)

print("labels moved and YOLO_LABEL_DIR removed.")

# ================= 新增的部分 =================

# 定義 classes.txt 的來源與目標路徑
src_classes = "classes.txt" # 當下目錄下的 classes.txt
dst_train_dir = os.path.join(DATASET_DIR, "labels", "train")
dst_val_dir = os.path.join(DATASET_DIR, "labels", "val")

# 確保目標資料夾 train 和 val 存在 (避免 shutil.copy 找不到資料夾報錯)
os.makedirs(dst_train_dir, exist_ok=True)
os.makedirs(dst_val_dir, exist_ok=True)

# 4️⃣ 複製 classes.txt 到 train 與 val 資料夾
if os.path.exists(src_classes):
    shutil.copy(src_classes, dst_train_dir)
    shutil.copy(src_classes, dst_val_dir)
    print("classes.txt successfully copied to train and val directories.")
else:
    print(f"⚠️ 警告：在當下目錄找不到 {src_classes}，略過複製步驟。")

## 開始訓練

### YOLO Library

In [ ]:
# from ultralytics import YOLO
# import os

# model = YOLO(PRETRAIN_MODEL)  # load a pretrained model (recommended for training)

# val_classes = os.path.join(DATASET_DIR, "labels/val/classes.txt")
# train_classes = os.path.join(DATASET_DIR, "labels/train/classes.txt")

# if os.path.exists(train_classes) and os.path.exists(val_classes):
#     results = model.train(data="data.yaml", epochs=10, imgsz=640, batch=32)
# else:
#     raise FileNotFoundError(f"Missing required files or directories: {val_classes} {train_classes}")

TypeError: conv2d() received an invalid combination of arguments - got (ReLU, Parameter, NoneType, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, tuple of ints padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!ReLU!, !Parameter!, !NoneType!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, int)
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, str padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!ReLU!, !Parameter!, !NoneType!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, int)


### Costum YOLO Architecture

In [1]:
import yaml
import os

def load_yolo_data(yaml_path):
    # 檢查檔案是否存在
    if not os.path.exists(yaml_path):
        print(f"找不到檔案: {yaml_path}")
        return None

    with open(yaml_path, 'r', encoding='utf-8') as f:
        try:
            # 使用 safe_load 避免執行惡意代碼，這是處理 YAML 的標準做法
            data = yaml.safe_load(f)
            return data
        except yaml.YAMLError as exc:
            print(f"讀取 YAML 時發生錯誤: {exc}")
            return None

config = load_yolo_data(DATA_YAML)

if config:
    print("--- 成功讀取資料集配置 ---")
    
    # 1. 取得類別數量
    nc = config.get('nc', 0)
    print(f"類別數量 (nc): {nc}")
    
    # 2. 取得類別名稱列表
    names = config.get('names', [])
    print(f"類別名稱: {names}")
    
    # 3. 取得訓練/驗證路徑
    train_path = config.get('train', '未設定')
    val_path = config.get('val', '未設定')
    print(f"訓練集路徑: {train_path}")
    print(f"驗證集路徑: {val_path}")

    # 如果 names 是字典格式 {0: 'person', 1: 'dog'}，可以這樣處理
    if isinstance(names, dict):
        class_list = list(names.values())
        print(f"轉換後的類別列表: {class_list}")

NameError: name 'DATA_YAML' is not defined

In [20]:
from arch.yolo11 import yolo_v11_n
from torchinfo import summary

yolo_model = yolo_v11_n(num_classes=nc)

# input_size 格式為 (Batch_size, Channels, Height, Width)
# YOLO 通常使用 640x640 的輸入
summary(yolo_model, input_size=(1, 3, 640, 640))

Layer (type:depth-idx)                                            Output Shape              Param #
YOLO                                                              [1, 8, 8400]              --
├─DarkNet: 1-1                                                    [1, 128, 80, 80]          --
│    └─Sequential: 2-1                                            [1, 16, 320, 320]         --
│    │    └─Conv: 3-1                                             [1, 16, 320, 320]         464
│    └─Sequential: 2-2                                            [1, 64, 160, 160]         --
│    │    └─Conv: 3-2                                             [1, 32, 160, 160]         4,672
│    │    └─CSP: 3-3                                              [1, 64, 160, 160]         6,640
│    └─Sequential: 2-3                                            [1, 128, 80, 80]          --
│    │    └─Conv: 3-4                                             [1, 64, 80, 80]           36,992
│    │    └─CSP: 3-5              

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# ==========================================
# 1. 替換為該 Repo 的模型、資料集與損失函數
# ==========================================
# 以下為示意 Import，你需要根據 main.py 或 nets/ 內的實際名稱修改
# from nets.yolo11 import YOLOv11
# from utils.dataloader import YOLODataset
# from nets.loss import YOLOLoss 

def train_one_epoch(model, dataloader, optimizer, criterion, device, epoch):
    model.train()
    total_loss = 0
    
    for batch_idx, (images, targets) in enumerate(dataloader):
        images, targets = images.to(device), targets.to(device)
        
        # 前向傳播
        outputs = model(images)
        
        # 計算損失 (YOLO 的 Loss 通常包裝好了，直接丟入預測和標籤)
        loss = criterion(outputs, targets)
        
        # 反向傳播與參數更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # 每 10 個 batch 印出一次進度
        if batch_idx % 10 == 0:
            print(f"Train Epoch: {epoch} [{batch_idx}/{len(dataloader)}] \t Loss: {loss.item():.4f}")
            
    return total_loss / len(dataloader)

def test_model(model, dataloader, criterion, device):
    model.eval()
    test_loss = 0
    
    # 測試階段不需要計算梯度
    with torch.no_grad():
        for images, targets in dataloader:
            images, targets = images.to(device), targets.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, targets)
            test_loss += loss.item()
            
            # 這裡通常會呼叫 mAP 計算函數 (如 utils.metrics 裡的邏輯)
            # calculate_map(outputs, targets)
            
    avg_loss = test_loss / len(dataloader)
    print(f"\nTest set: Average loss: {avg_loss:.4f}\n")
    return avg_loss

if __name__ == "__main__":
    # 設定硬體設備
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # ==========================================
    # 2. 超參數設定與實例化
    # ==========================================
    epochs = 50
    batch_size = 16
    learning_rate = 1e-3
    
    """
    # 註解取消後，替換為該專案實際的資料集與模型呼叫方式
    
    # 準備 DataLoader
    train_dataset = YOLODataset(annotation_lines_train, input_shape, ...)
    test_dataset  = YOLODataset(annotation_lines_test, input_shape, ...)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # 初始化模型與損失函數
    model = YOLOv11(num_classes=80).to(device)
    criterion = YOLOLoss()
    
    # 設定優化器
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # ==========================================
    # 3. 執行主迴圈
    # ==========================================
    for epoch in range(1, epochs + 1):
        print(f"--- Starting Epoch {epoch} ---")
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
        test_loss = test_model(model, test_loader, criterion, device)
        
        # 可以加上儲存權重的邏輯
        # torch.save(model.state_dict(), f"logs/ep{epoch:03d}-loss{train_loss:.3f}.pth")
    """